## Background 

A new Premier League season has begun, and that means millions of fans are gearing up to play Fantasy Premier League (FPL) again, which is a fantasy football game where users select 15 players from the Premier League whilst balancing a fixed budget, earning points based on real-world match performance. 

After exploring introductory optimisation problems in first-year, I found out that FPL actually has neat parallels to portfolio construction and optimisation in finance: selecting assets from a large universe, subject to multiple constraints to maximise expected returns. There are a few similarities in both being a constrained optimisation problem:  

- A manager selecting a range of **players** from a large universe of choices -> An investor selecting a range of **assets** from a large universe of choices. 
- A manager maximising expected **points** subject to strict in-game constraints -> An investor maximising expected **returns** subject to real-life financial constraints. 

The constraints themselves have some similarities,  like these three examples:

1. An FPL constraint like the fixed £100m budget is similar to the maximum available capital that investors have. 
2. No more than three players from the same team are allowed, introducing a diversification constraint similar to portfolio construction.
3. Neither problem is deterministic: future player performance is uncertain (injuries, drops in form, harder matchdays ahead), just like the uncertainty of asset returns (associated risk, market downturns).

The final point raises a key question: how should a manager balance expected performance against uncertainty? This project applies portfolio optimisation techniques to FPL team selection, aiming to test whether a systematic, risk-aware method produces better teams than greedy player picking. 

## Question

"Can portfolio optimisation techniques improve FPL team selection compared to simple greedy-picking heuristics?"

## Modern Portfolio Theory and FPL

Modern Portfolio Theory (MPT), developed by Harry Markowitz in 1952, provides a framework for constructing portfolios by considering expected return, risk, and how assets move together through their covariances.

In this project, we take MPT's use for financial assets and apply it to our FPL team. Instead of selecting assets to maximise expected returns, we select players to maximise expected points. However, a player's individual value should not be considered in isolation; uncertainty of points and how their performance impacts other players in the team must also be considered. 

This leads to the central idea behind the project (and MPT in general): The best team is not necessarily the team containing the 15 players with the highest predicted points. That is actually known as 'greedy selection,' which forms the baseline of this project. Instead, portfolio optimisation aims to combine players that achieve the best trade-off between expected points and risk. 

Consider three Arsenal defenders in our team. Arguably, they have the strongest defence in the league, so they may give strong expected points. However, this is not guaranteed. They could concede, or one may get a red card; as a result, all of them lose out together. Selecting players whose performances are less correlated can therefore provide greater diversification and potentially reduce overall team risk. This is why FPL forces players to not choose more than three players from the same team. 

In financial portfolio optimisation, this trade-off can be expressed mathematically as objective functions: either maximising expected return for a given level of risk, or minimising risk for a given level of return. We apply the same principle to FPL by seeking teams that maximise expected points while factoring in player volatility and covariance.

The following sections formalise these ideas and translate them into an FPL optimisation problem.

### Expected Return

In MPT, each asset has an expected return $\mu_i$, usually written as:

$$
\mu_i = \mathbb{E}[R_i]
$$

where $R_i$ is the return of asset $i$. 

For FPL, we can replace return with player points, denoted by $P_i$. If we estimate that a player will score 6 points in the next gameweek, then their expected points are:

$$
\mu_i = \mathbb{E}[P_i] = 6
$$

For a portfolio, expected return is simply the weighted sum of the individual expected returns:

$$
\mathbb{E}[R_p] = \sum_{i=1}^{n} w_i\mu_i
$$

where $w_i$ is the weight of asset $i$ in the portfolio (the proportion of the total portfolio invested in that asset).

A key distinction between this and our FPL model is that in a traditional portfolio, you choose how much to invest in that asset, so $w_i$ is a continuous value. However, in FPL you either select a player or you don't, which is a binary selection instead. This is expanded upon as we build the optimisation problem itself. The intuition remains same for FPL: expected points of a team depend on the expected points of the players selected. However, expected points are predictions, not guarantees. A player expected to score 6 points could score 2, 10, or 0. This uncertainty leads us to risk.

### Volatility / Risk

In MPT, individual asset risk is measured by the variance of their returns, or its square root, the standard deviation. For FPL, we can apply the same idea for individual player risk: 

$$
\sigma_i^2 = \operatorname{Var}(P_i)
$$

$$
\sigma_i = \sqrt{\operatorname{Var}(P_i)}
$$

where $\sigma_i$ is the individual player risk. To explain how portfolio risk works, consider two players A and B: 

- **Player A:** 2, 8, 2, 8, 2, 8
- **Player B:** 5, 6, 5, 6, 5, 6

The average points are similar, but Player A's points variation is much greater than Player B, who has a more consistent spread. For this reason, a manager would prefer players that provide smaller, predictable returns like B rather than larger, volatile ones like A.

However, portfolio risk as a whole is not simply the sum of individual player risks. This is where covariance and correlation are introduced. 

### Covariance and Correlation

So far, we have calculated variances of assets, but this only tells us how returns move individually - not together. That's where covariance comes in. For the FPL scenario, covariance, $\operatorname{Cov}(P_i,P_j)$ measures the tendency of two players' points to move together across gameweeks.

Using the previous example, positive covariance means that Player A's points being above average implies the same for Player B. Negative covariance means their performances tend to move in opposite directions. We can also standardise covariance, also known as correlation:

$$
\rho_{ij}
=
\frac{\operatorname{Cov}(P_i,P_j)}
{\sigma_i\sigma_j}
$$

with values between -1 and 1.

The FPL interpretation is that two defenders from the same team may have $\rho_{ij}$ close to 1 because a clean sheet can benefit both players simultaneously. Meanwhile, players from completely different teams and positions may have $\rho_{ij}$ closer to 0 instead. Note that we don't assume these relationships - it is simply estimated from historical FPL data.

This is another key insight from MPT: diversification works because portfolio assets do not necessarily move perfectly together. The same principle potentially applies to an FPL team. 

### Portfolio Risk

Now that we have defined the key components of portfolio theory (expected return, individual risk, covariance between assets), we can combine these ideas to calculate the overall risk of a portfolio.

We start from the portfolio variance. By definition of variance in terms of covariance, we have:

$$
\sigma_p^2
=
\sum_{i=1}^{n}\sum_{j=1}^{n}
w_i w_j \operatorname{Cov}(P_i,P_j)
$$

This says to take every possible pair of assets, see how they move together (covariance), and weigh that relationship depending on how much of each asset you hold ($w_iw_j$). 

The double sum does this for every possible pair of assets, including when $i = j$, which is just $Cov(P_i,P_i) = Var(P_i)$. This is useful for the following covariance matrix that we also define:

$$
\Sigma =
\begin{bmatrix}
\operatorname{Cov}(P_1,P_1) & \operatorname{Cov}(P_1,P_2) & \cdots & \operatorname{Cov}(P_1,P_n) \\
\operatorname{Cov}(P_2,P_1) & \operatorname{Cov}(P_2,P_2) & \cdots & \operatorname{Cov}(P_2,P_n) \\
\vdots & \vdots & \ddots & \vdots \\
\operatorname{Cov}(P_n,P_1) & \operatorname{Cov}(P_n,P_2) & \cdots & \operatorname{Cov}(P_n,P_n)
\end{bmatrix}
$$

This is an $n * n$ matrix containing every possible relationship between assets. Note how the diagonals contain individual variances (individual volatility per player), whilst the non-diagonals contain pairwise covariances (how players move together). So to summarise, the covariance matrix sums up individual risks and relationships between risks. 

Next, we define the vector of portfolio weights:

$$
\mathbf{w}
=
\begin{bmatrix}
w_1 \\
w_2 \\
\vdots \\
w_n
\end{bmatrix}.
$$

This is what was described in expected return, this time in vector form. Now we have $\mathbf{w}$ (what you hold), $\Sigma$ (how everything behaves), and $\sigma_p^2$ (how risky the resulting portfolio is).  

Consider the simplest example: the two-player case. The possible combinations are $(i,j) = (1,1), (1,2), (2,1), (2,2)$. Therefore, applying the portfolio variance formula gives:

$$
\sigma_p^2
=
w_1^2\operatorname{Cov}(P_1,P_1)
+
w_1w_2\operatorname{Cov}(P_1,P_2)
+
w_2w_1\operatorname{Cov}(P_2,P_1)
+
w_2^2\operatorname{Cov}(P_2,P_2).
$$

Using the following covariance properties (variance and symmetry): 

$$\operatorname{Cov}(P_i,P_i)=\operatorname{Var}(P_i)$$ and $$\operatorname{Cov}(P_1,P_2)=\operatorname{Cov}(P_2,P_1)$$

we obtain:

$$
\sigma_p^2
=
w_1^2\operatorname{Var}(P_1)
+
w_2^2\operatorname{Var}(P_2)
+
2w_1w_2\operatorname{Cov}(P_1,P_2).
$$

The same calculation extends to any number of assets. 

Therefore, the double sum can be written compactly in matrix form as:

$$
\sigma_p^2
=
\mathbf{w}^{\top}\Sigma\mathbf{w}
$$

and portfolio volatility is:

$$
\sigma_p
=
\sqrt{\mathbf{w}^{\top}\Sigma\mathbf{w}}
.
$$

The distinction in FPL is that we replace portfolio weights $\mathbf{w}$ with a binary selection vector $\mathbf{x}$, where $x_i = 1$ if player $i$ is selected and $x_i = 0$ otherwise. The asset covariance matrix $\Sigma$ becomes the covariance matrix of player points. The formula becomes:

$$
\sigma_{\text{team}}^2 = \mathbf{x}^\top\Sigma\mathbf{x}
$$

This means team variance has two components, similarly to portfolio variance:

1. Individual volatility (diagonal terms): $\sum_{i=1}^{n} y_i \operatorname{Var}(P_i)$ - how much each starting player's points fluctuate around their average.

2. Co-movement (off-diagonal terms): $\sum_{i \neq j} y_i y_j \operatorname{Cov}(P_i, P_j)$ - how much the starting players' points move together.

If all 11 starting players are highly correlated (e.g., all from strong teams), their points tend to move together, and team variance is high. Conversely, if selected players have low or negative correlations, the team variance is lower for the same expected points.

This is the quantitative case for diversification: selecting players whose performance correlates less will reduce overall team risk.

### Squad vs. Starting XI

So far, $\mathbf{x}$ has represented the full 15-player squad. However, only 11 of those players actually start and score points each gameweek; the remaining 4 sit on the bench. To capture this, we introduce a second binary variable:

$$
y_i =
\begin{cases}
1, & \text{if player } i \text{ starts the gameweek},\\
0, & \text{otherwise}.
\end{cases}
$$

A player can only start if they are already in the squad, so the two variables are linked by:

$$
y_i \leq x_i \qquad \forall i.
$$

This prevents a logical contradiction: if $x_i = 0$ but $y_i = 1$, a player is in the starting XI despite not even being in the 15-man squad - the constraint rules this out. 

Exactly 11 players start each gameweek:

$$
\sum_{i=1}^{n} y_i = 11.
$$

Since expected points and risk are only realised by the starting XI, portfolio risk and expected points should be calculated using $\mathbf{y}$, not $\mathbf{x}$. Important note that this project deliberately excludes Bench Boost (the chip that scores all 15 players in a given gameweek), as well as other chips like Triple Captain (triples captain points), Wild Card (unlimited, free permanent transfers) and Free Hit (unlimited free transfers for one gameweek), since they are not used every week. The addition of chips are definitely something worth looking at should I wish to develop this project further, but for now the scope remains relatively minimal.

Here is the 

$$
\mathbb{E}[P_{\text{XI}}] = \mathbf{y}^{\top}\boldsymbol{\mu}, \qquad \sigma_{\text{XI}}^2 = \mathbf{y}^{\top}\Sigma\mathbf{y}.
$$

Squad selection ($\mathbf{x}$) and starting XI selection ($\mathbf{y}$) are optimised sequentially rather than jointly: the squad is chosen first, then the strongest XI is selected from within it. A joint optimisation would likely find a better combined outcome, since it wouldn't be constrained by a squad chosen without knowledge of the eventual XI split, but sequential optimisation keeps the project clearly explainable within a second-year scope, rather than adding complexity that stalls progress for a first attempt at this kind of problem.

### The Return–Risk Trade-off

MPT now presents us with two competing objectives:

1. Maximise expected points
2. Minimise portfolio risk

Note that both objectives do not necessarily produce the same team: a team containing more top performers may have a higher expected score but also greater uncertainty. Another team might sacrifice some expected points in exchange for a lower portfolio variance.

Now that we have these two objectives, let's construct an objective function for the first one. The key idea is to combine the two quantities already derived.

Expected portfolio return (in vector form):

$$
\mathbb{E}[R_p]
=
\mathbf{w}^{\top}\boldsymbol{\mu}
$$

Portfolio variance (risk):

$$
\sigma_p^2
=
\mathbf{w}^{\top}\Sigma\mathbf{w}
$$

As mentioned before, we don't necessarily want to maximise one expression while ignoring the other; we want some balance between both. MPT balances these by maximising the difference between risk and reward, with an additional constant $\lambda$ acting as a conversion parameter between both objectives: 

$$
\max_{\mathbf{w}}
\quad
\mathbf{w}^{\top}\boldsymbol{\mu}
-
\lambda\mathbf{w}^{\top}\Sigma\mathbf{w}.
$$

Here, $\lambda\geq0$ controls the risk-return trade-off. In other words, it determines how heavy the optimiser penalises variance: 

- If $\lambda = 0$, then risk is completely ignored and we solely focus on maximising expected points. 
- If $\lambda$ is small, risk is lightly penalised, so expected return is still prioritised.
- If $\lambda$ is high, risk is highly penalised, so the optimiser is willing to sacrifice expected points for a lower-risk team.

For FPL, we replace the portfolio weights $\mathbf{w}$ with the binary selection vector $\mathbf{x}$:

$$
\mathbb{E}[P_{\text{team}}]
=
\mathbf{x}^{\top}\boldsymbol{\mu}
$$

and

$$
\sigma_{\text{team}}^2
=
\mathbf{x}^{\top}\Sigma\mathbf{x}.
$$

Therefore, the FPL mean-variance objective becomes:

$$
\max_{\mathbf{x}}
\quad
\mathbf{x}^{\top}\boldsymbol{\mu}
-
\lambda\mathbf{x}^{\top}\Sigma\mathbf{x}
$$

subject to the FPL constraints. Intuitively this makes no difference to MPT itself: just a notation switch occurs.  

Now the FPL problem is more than choosing players with the highest predicted points; the optimiser now considers how the players work together as a portfolio.

### Sharpe Ratio

A key performance metric in evaluating portfolio performance is the Sharpe ratio, which measures excess return relative to a risk-free rate against overall volatility:

$$
S = \frac{R_p-R_f}{\sigma_p}
$$

For FPL, we omit the risk-free rate as there is no obvious equivalent. Hence this simplifies to:

$$
S_{\text{FPL}} \approx \frac{P_p}{\sigma_p}
$$

This can be interpreted as points generated per unit of portfolio risk.

It gives us another way of comparing teams. A team with slightly fewer expected points could still be attractive if it achieves those points with substantially lower risk.

### Efficient Frontier

The efficient frontier in MPT represents the set of portfolios offering the best possible expected return for each level of risk. Applying this to FPL, imagine generating many different valid teams: some have high expected points but high risk, while others have low expected points with lower risk. The efficient teams are those for which there is no alternative team that:

- has higher expected points for the same level of risk, or
- has lower risk for the same expected points.

Plotting these teams graphically gives us the efficient frontier. This gives a more conclusive way of thinking about the final optimisation problem. Instead of assuming just one optimal FPL team, we can investigate the trade-off between expected points and portfolio risk before seeing how the optimal team changes with our tolerance for risk. Now we begin to formulate the FPL problem itself.


## FPL Problem Formulation

Now that the MPT framework has been established, we now translate the FPL selection problem into a formal optimisation problem.

Suppose there are $n$ available players across all of FPL. For each player $i$, we define a binary decision variable:

$$
x_i =
\begin{cases}
1, & \text{if player } i \text{ is selected},\\
0, & \text{otherwise}.
\end{cases}
$$

The vector

$$
\mathbf{x}=(x_1,x_2,\ldots,x_n)^\top
$$

therefore represents the composition of the FPL squad. 

Each player has their own price $c_i$, position, club, and estimated performance characteristics. The optimisation problem is to determine the values of $x_i$ that produce the best possible squad that simultaenously satisfies the rules of the game.

### FPL Constraints

#### Budget constraint

Let $c_i$ denote the price of player $i$. The total squad cost cannot exceed the available budget $B$ (in millions of pounds), which is 100. Therefore, we have:

$$
\sum_{i=1}^{n}c_i x_i\leq 100
$$

#### Squad size

An FPL squad must contain exactly 15 players:

$$
\sum_{i=1}^{n}x_i=15
$$

#### Position constraints

Let $G_i,D_i,M_i,F_i$ be indicator variables taking the value 1 when player $i$ is respectively a goalkeeper, defender, midfielder, or forward.

The squad must contain exactly two goalkeepers:

$$
\sum_{i=1}^{n}G_i x_i=2
$$

five defenders:

$$
\sum_{i=1}^{n}D_i x_i=5
$$

five midfielders:

$$
\sum_{i=1}^{n}M_i x_i=5
$$

and three forwards:

$$
\sum_{i=1}^{n}F_i x_i=3.
$$

#### Club constraint

An FPL squad can contain no more than three players from the same Premier League club. Let $C_{ik}=1$ if player $i$ belongs to club $k$, and 0 otherwise. Then:

$$
\sum_{i=1}^{n}C_{ik}x_i\leq3
\qquad \forall k = 1, ... , 20.
$$

Finally, each player is either selected or not selected:

$$
x_i\in\{0,1\}
\qquad \forall i = 1, ... , n.
$$

Together, these constraints define the feasible set of FPL squads.

### Optimisation Objectives

With the feasible set established, now we ask what an "optimal" team actually means. This project considers three increasingly sophisticated approaches to answer this question. 

#### Objective 1: Greedy Selection

A greedy heuristic is the most straightforward, therefore acting as the benchmark against which the other two optimisation-based approaches are tested on. Here, players are ranked by a performance measure like predicted points, then selected sequentially subject to FPL constraints. 

This mirrors someone asking, 'which player gives me the most value from remaining options?' However, greedy selection does not necessarily form the most optimal squad, as locally attractive choices prevent better overall combinations from being used. That's where the other two optimisation methods come in. 

#### Objective 2: Mean-Variance Optimisation

The first portfolio-based objective follows MPT by balancing expected points against portfolio risk:

$$
\max_{\mathbf{x}}
\quad
\boldsymbol{\mu}^{\top}\mathbf{x}
-
\lambda\mathbf{x}^{\top}\Sigma\mathbf{x}
$$

subject to the FPL constraints above, where $\lambda \geq 0$ controls the trade-off between expected points and risk.

#### Objective 3: Balanced Optimisation

The third approach also uses expected points and risk, but notice how both variables are on different scales and units: variance is more likely to dominate expected points since it's a squared term, therefore its numbers are larger. In Objective 2, this issue was controlled by $\lambda$: not only did it control the trade-off between expected points and risk, but it scaled the variance term in the objective function. 

For this objective, we normalise both expected points and risk to a dimensionless scale with interval $[0,1]$.

For expected points:

$$
\tilde{\mu}_p
=
\frac{\mu_p-\mu_{\min}}
{\mu_{\max}-\mu_{\min}}
$$

and for portfolio variance:

$$
\tilde{\sigma}_p^2
=
\frac{\sigma_p^2-\sigma_{\min}^2}
{\sigma_{\max}^2-\sigma_{\min}^2}.
$$

The two normalised quantities can then be combined using:

$$
\max_{\mathbf{x}}
\quad
\alpha\tilde{\mu}_p
-
(1-\alpha)\tilde{\sigma}_p^2
$$

where:

$$
0\leq\alpha\leq1.
$$

Here, $\alpha$ directly controls the relative importance of expected points and risk:

- $\alpha=1$: maximise expected points only.
- $\alpha=0.5$: give equal weight to expected points and risk.
- $\alpha=0$: minimise risk only.

### Estimating the Model Parameters

The optimisation itself depends on estimating three parameters from historical FPL data:

1. Expected points $\mu_i$ for each player: using recent form, fixture difficulty, and player statistics.
2. Volatility $\sigma_i$ for each player: from the standard deviation of historical gameweek points.
3. Covariance matrix $\Sigma$: from historical correlations between player points.

From this, the empirical question is whether these portfolio-informed objectives produce better FPL teams than the greedy baseline.